# 05_modeling.ipynb

4주차 B팀 모델링 코드입니다. 입력 파일은 3주차 결과물인 `modeling_dataset.csv`입니다. `monthly_merged.csv`를 다시 불러오지 않습니다.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
# 1. modeling_dataset.csv 불러오기
# 이 노트북은 notebooks 폴더에서 실행하는 것을 기준으로 작성했습니다.
# 다른 위치에서 실행해도 찾을 수 있도록 후보 경로를 여러 개 둡니다.

candidate_paths = [
    Path("../data/processed/modeling_dataset.csv"),
    Path("data/processed/modeling_dataset.csv"),
    Path("../processed/modeling_dataset.csv"),
    Path("modeling_dataset.csv"),
]

input_path = None
for path in candidate_paths:
    if path.exists():
        input_path = path
        break

if input_path is None:
    raise FileNotFoundError("modeling_dataset.csv를 찾을 수 없습니다. data/processed 폴더 위치를 확인하세요.")

print("사용 데이터:", input_path)
df = pd.read_csv(input_path)
df = df.sort_values(["gu", "contract_month"]).reset_index(drop=True)

print(df.shape)
df.head()

In [ ]:
# 2. 필요한 컬럼 확인
required_cols = [
    "gu", "contract_month", "jeonse_rate", "gap_rate",
    "sale_growth_1m", "jeonse_growth_1m", "growth_gap_1m",
    "sale_volume_growth_1m", "rent_volume_growth_1m", "monthly_ratio",
    "user_risk_score", "investor_risk_score", "total_risk_score",
    "risk_target", "warning_flag"
]

missing_cols = [col for col in required_cols if col not in df.columns]
print("누락 컬럼:", missing_cols)

if missing_cols:
    raise ValueError(f"필요한 컬럼이 없습니다: {missing_cols}")

In [ ]:
# 3. 최종 통합 정리본 용어와 맞추기
# 3주차 데이터셋의 user_risk_score / investor_risk_score를
# 최종 문서의 resident_risk_index / investor_risk_index 용어로 함께 사용합니다.

df["resident_risk_index"] = df["user_risk_score"]
df["investor_risk_index"] = df["investor_risk_score"]

df["resident_risk_grade"] = df.get("user_risk_grade")
df["investor_risk_grade"] = df.get("investor_risk_grade")

# monthly_ratio_diff는 modeling_dataset.csv에 있는 monthly_ratio만으로 계산 가능합니다.
df["monthly_ratio_diff"] = df.groupby("gu")["monthly_ratio"].diff().fillna(0)
df["monthly_shift_abs"] = df["monthly_ratio_diff"].abs()

df[["resident_risk_index", "investor_risk_index", "monthly_ratio_diff"]].head()

In [ ]:
# 4. 정체도 보조 지표 생성
# 최종 통합 정리본의 정체도 5단계 기준을 modeling_dataset.csv의 기존 컬럼으로 생성합니다.

gap_q75 = df["gap_rate"].quantile(0.75)
monthly_shift_q75 = df["monthly_shift_abs"].quantile(0.75)

df["sale_stagnation_flag"] = (df["sale_growth_1m"].abs() <= 0.01).astype(int)
df["jeonse_stagnation_flag"] = (df["jeonse_growth_1m"].abs() <= 0.01).astype(int)
df["volume_drop_flag"] = (df["sale_volume_growth_1m"] < 0).astype(int)
df["high_gap_flag"] = (df["gap_rate"] >= gap_q75).astype(int)
df["rent_shift_flag"] = (df["monthly_shift_abs"] >= monthly_shift_q75).astype(int)

df["stagnation_score"] = (
    0.25 * df["sale_stagnation_flag"] +
    0.20 * df["jeonse_stagnation_flag"] +
    0.25 * df["volume_drop_flag"] +
    0.20 * df["high_gap_flag"] +
    0.10 * df["rent_shift_flag"]
)

def stagnation_level(score):
    if score <= 0.20:
        return "1단계_활발관찰"
    elif score <= 0.40:
        return "2단계_완만정체"
    elif score <= 0.60:
        return "3단계_안정정체"
    elif score <= 0.80:
        return "4단계_신중정체"
    else:
        return "5단계_거래위축정체"

df["stagnation_level"] = df["stagnation_score"].apply(stagnation_level)

df["stagnation_level"].value_counts()

In [ ]:
# 5. 모델 입력 변수 설정
# 위험 점수 자체를 입력으로 쓰면 risk_target과 너무 직접적으로 연결될 수 있으므로,
# 모델 입력에는 기본 시장 지표를 사용합니다.

features = [
    "jeonse_rate",
    "gap_rate",
    "sale_growth_1m",
    "jeonse_growth_1m",
    "growth_gap_1m",
    "sale_volume_growth_1m",
    "rent_volume_growth_1m",
    "monthly_ratio",
]

target = "risk_target"

X = df[features].copy()
y = df[target].astype(int)

print(X.shape, y.shape)
print(y.value_counts())

In [ ]:
# 6. 데이터 표준화 및 K-Means 군집화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df["kmeans_cluster"] = kmeans.fit_predict(X_scaled)

df["kmeans_cluster"].value_counts().sort_index()

In [ ]:
# 7. 로지스틱 회귀와 랜덤 포레스트 모델 적용
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

random_forest_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
}

result_rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    result_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1_score": f1_score(y_test, pred, zero_division=0),
    })
    
    pred_col = name.lower().replace(" ", "_") + "_pred"
    prob_col = name.lower().replace(" ", "_") + "_risk_proba"
    df[pred_col] = model.predict(X)
    if hasattr(model, "predict_proba"):
        df[prob_col] = model.predict_proba(X)[:, 1]

model_comparison = pd.DataFrame(result_rows)
model_comparison

In [ ]:
# 8. 변수 중요도 분석
feature_importance = pd.DataFrame({
    "feature": features,
    "importance": random_forest_model.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

feature_importance

In [ ]:
# 9. K-Means 군집 요약
cluster_summary = df.groupby("kmeans_cluster").agg(
    sample_count=("gu", "count"),
    avg_total_risk_score=("total_risk_score", "mean"),
    avg_resident_risk_index=("resident_risk_index", "mean"),
    avg_investor_risk_index=("investor_risk_index", "mean"),
    avg_stagnation_score=("stagnation_score", "mean"),
    avg_gap_rate=("gap_rate", "mean"),
    avg_sale_growth_1m=("sale_growth_1m", "mean"),
    avg_growth_gap_1m=("growth_gap_1m", "mean"),
    avg_monthly_ratio=("monthly_ratio", "mean"),
    risk_target_rate=("risk_target", "mean"),
    warning_rate=("warning_flag", "mean")
).reset_index()

cluster_summary["risk_rank"] = cluster_summary["avg_total_risk_score"].rank(method="first").astype(int)
rank_to_label = {
    1: "안정권",
    2: "양호",
    3: "관찰",
    4: "신중검토",
    5: "우선점검"
}
cluster_summary["cluster_risk_label"] = cluster_summary["risk_rank"].map(rank_to_label)
cluster_summary = cluster_summary.sort_values("avg_total_risk_score", ascending=False).reset_index(drop=True)
cluster_summary

In [ ]:
# 10. 최신 월 기준 지역별 위험 순위
latest_month = df["contract_month"].max()
latest_region_risk_ranking = df[df["contract_month"] == latest_month].copy()

ranking_cols = [
    "gu", "contract_month", "total_risk_score", "resident_risk_index", "investor_risk_index",
    "risk_grade", "resident_risk_grade", "investor_risk_grade", "warning_flag", "kmeans_cluster",
    "stagnation_score", "stagnation_level", "gap_rate", "sale_growth_1m", "growth_gap_1m", "monthly_ratio"
]
ranking_cols = [c for c in ranking_cols if c in latest_region_risk_ranking.columns]

latest_region_risk_ranking = latest_region_risk_ranking[ranking_cols].sort_values(
    "total_risk_score", ascending=False
).reset_index(drop=True)
latest_region_risk_ranking.insert(0, "rank", range(1, len(latest_region_risk_ranking) + 1))

latest_region_risk_ranking.head(10)

In [ ]:
# 11. 결과 저장
out_dir = Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)

model_comparison.to_csv(out_dir / "model_comparison.csv", index=False, encoding="utf-8-sig")
feature_importance.to_csv(out_dir / "feature_importance.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv(out_dir / "kmeans_cluster_summary.csv", index=False, encoding="utf-8-sig")
latest_region_risk_ranking.to_csv(out_dir / "latest_region_risk_ranking.csv", index=False, encoding="utf-8-sig")
df.to_csv(out_dir / "modeling_result_dataset.csv", index=False, encoding="utf-8-sig")

print("저장 완료:", out_dir)